In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .config("spark.jars", "/app/code/my_udf/target/my-udfs.jar")   
         .getOrCreate())

# register once per session; the name "sanitize_ip" comes from the Java file
spark.sparkContext._jvm.com.techlock.udfs.SanitizeIPUdf.initSanitizeIP(spark._jsparkSession)

spark.sql("SELECT sanitize_ip('8.8.8.8')").show()

# sample DataFrame to test the UDF on a column
df = spark.createDataFrame([("8.8.8.8",), ("not-an-ip",), ("2001:db8::1",)], ["source_ip"])
df.selectExpr("source_ip", "sanitize_ip(source_ip) AS ip").show()

26/10/06 13:50:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 13:50:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+--------------------+
|sanitize_ip(8.8.8.8)|
+--------------------+
|             8.8.8.8|
+--------------------+



+-----------+-----------+
|  source_ip|         ip|
+-----------+-----------+
|    8.8.8.8|    8.8.8.8|
|  not-an-ip|       NULL|
|2001:db8::1|2001:db8::1|
+-----------+-----------+

